# Module 00 — Notebook 01: Environment Setup

## Learning Objectives
By completing this notebook, you will:
1. Verify that Captum, PyTorch, torchvision, and transformers are installed correctly
2. Confirm GPU or CPU availability and understand performance implications
3. Download and verify pretrained models from torchvision and HuggingFace
4. Run a smoke test confirming the full attribution pipeline works end-to-end

## Prerequisites
- Python 3.9+
- pip or conda package manager
- Internet connection for model downloads

## Estimated Time: 10 minutes

---

## 1. Install Required Packages

Run the cell below if any packages are missing. If you are on a managed environment (Google Colab, AWS SageMaker, etc.), these may already be available.

In [ ]:
# Install or upgrade required packages
# Remove the # to run this cell if packages are not already installed

# !pip install captum>=0.7.0 torch>=2.0.0 torchvision>=0.15.0 transformers>=4.30.0
# !pip install matplotlib seaborn Pillow requests tqdm

print("Run the pip install line above if packages are missing.")
print("Then restart your kernel and re-run this notebook.")

## 2. Version Verification

The cell below checks all package versions and reports the hardware configuration. 
Expected output: all packages present, version numbers displayed.

In [ ]:
# Verify package versions and hardware
import sys
import platform

print("=" * 60)
print("ENVIRONMENT VERIFICATION")
print("=" * 60)

# Python version
print(f"\nPython: {sys.version}")
print(f"Platform: {platform.platform()}")

# PyTorch
try:
    import torch
    print(f"\nPyTorch: {torch.__version__}")
    print(f"  CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"  CUDA version: {torch.version.cuda}")
        print(f"  GPU: {torch.cuda.get_device_name(0)}")
        print(f"  GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    else:
        print("  Running on CPU — attributions will be slower but fully functional")
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"  Active device: {DEVICE}")
except ImportError:
    print("ERROR: PyTorch not found. Run: pip install torch")

# torchvision
try:
    import torchvision
    print(f"\ntorchvision: {torchvision.__version__}")
except ImportError:
    print("\nERROR: torchvision not found. Run: pip install torchvision")

# Captum
try:
    import captum
    print(f"\nCaptum: {captum.__version__}")
except ImportError:
    print("\nERROR: Captum not found. Run: pip install captum")

# Transformers
try:
    import transformers
    print(f"\ntransformers: {transformers.__version__}")
except ImportError:
    print("\nERROR: transformers not found. Run: pip install transformers")

# Visualization
try:
    import matplotlib
    import seaborn
    print(f"\nmatplotlib: {matplotlib.__version__}")
    print(f"seaborn: {seaborn.__version__}")
except ImportError as e:
    print(f"\nVisualization error: {e}")

# PIL
try:
    from PIL import Image
    import PIL
    print(f"\nPillow: {PIL.__version__}")
except ImportError:
    print("\nERROR: Pillow not found. Run: pip install Pillow")

print("\n" + "=" * 60)
print("Verification complete.")
print("=" * 60)

## 3. Download Pretrained Models

This course uses pretrained models from two sources:

1. **torchvision** — ImageNet-pretrained CNNs (ResNet, VGG, EfficientNet)
2. **HuggingFace Hub** — Pretrained transformers (DistilBERT for sentiment)

The first download may take 1–5 minutes depending on your connection. Models are cached locally after the first download.

In [ ]:
# Download and verify torchvision pretrained models
import torch
import torchvision.models as models

print("Downloading pretrained torchvision models...")
print("(This downloads weights from PyTorch Hub — first run only)\n")

# ResNet-50: the primary model for image attribution demos
# weights='IMAGENET1K_V1' downloads ImageNet-pretrained weights
resnet50 = models.resnet50(weights='IMAGENET1K_V1')
resnet50.eval()  # CRITICAL: always set to eval mode for attribution
print("ResNet-50: LOADED")
print(f"  Parameters: {sum(p.numel() for p in resnet50.parameters()):,}")

# VGG-16: used for GradCAM demos (has named feature layers)
vgg16 = models.vgg16(weights='IMAGENET1K_V1')
vgg16.eval()
print("\nVGG-16: LOADED")
print(f"  Parameters: {sum(p.numel() for p in vgg16.parameters()):,}")

print("\nModel downloads complete.")

In [ ]:
# Download and verify HuggingFace pretrained model
# Using DistilBERT fine-tuned on SST-2 (movie review sentiment)
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_NAME = "distilbert-base-uncased-finetuned-sst-2-english"

print(f"Downloading {MODEL_NAME}...")
print("(~250MB download from HuggingFace Hub — first run only)\n")

# Download tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
bert_model.eval()

print(f"DistilBERT ({MODEL_NAME}): LOADED")
print(f"  Parameters: {sum(p.numel() for p in bert_model.parameters()):,}")
print(f"  Labels: {bert_model.config.id2label}")

# Quick smoke test
test_text = "This movie was absolutely wonderful!"
inputs = tokenizer(test_text, return_tensors="pt")

with torch.no_grad():
    outputs = bert_model(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1)

label = bert_model.config.id2label[probs.argmax().item()]
confidence = probs.max().item()

print(f"\nSmoke test: '{test_text}'")
print(f"  Prediction: {label} ({confidence:.1%} confidence)")
print("  Expected: POSITIVE")

## 4. ImageNet Class Labels

For image classification explanations, we need ImageNet class names to understand what the model predicted. We download the standard 1000-class label list.

In [ ]:
# Download ImageNet class labels
import json
import urllib.request

IMAGENET_LABELS_URL = (
    "https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels"
    "/master/imagenet-simple-labels.json"
)

print("Downloading ImageNet class labels...")

try:
    with urllib.request.urlopen(IMAGENET_LABELS_URL) as response:
        imagenet_labels = json.loads(response.read().decode())
    print(f"Loaded {len(imagenet_labels)} class labels")
    print(f"\nSample labels:")
    for i in [0, 1, 281, 282, 483, 500]:
        print(f"  Class {i:4d}: {imagenet_labels[i]}")
except Exception as e:
    # Fallback: create a small label dictionary for the demo classes we use
    print(f"Could not download labels ({e}). Using fallback labels.")
    imagenet_labels = [f"class_{i}" for i in range(1000)]
    # Manually set common classes
    imagenet_labels[281] = "tabby cat"
    imagenet_labels[282] = "tiger cat"
    imagenet_labels[243] = "bull mastiff"
    imagenet_labels[386] = "African elephant"
    imagenet_labels[483] = "castle"

print("\nImageNet labels ready.")

## 5. Test Image Preprocessing Pipeline

Every image attribution in this course uses the standard ImageNet preprocessing pipeline. The cell below verifies it works correctly and shows what the preprocessing does to an input image.

In [ ]:
# Verify image preprocessing pipeline
import torch
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import urllib.request
import io

# Standard ImageNet preprocessing
# Mean and std from the ImageNet dataset
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

preprocess = transforms.Compose([
    transforms.Resize(256),          # Resize shorter side to 256
    transforms.CenterCrop(224),      # Crop center 224×224
    transforms.ToTensor(),           # Convert to [0, 1] tensor
    transforms.Normalize(            # Normalize with ImageNet stats
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

# Inverse transform for visualization (undo normalization)
def denormalize(tensor):
    """Convert normalized tensor back to displayable image."""
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return torch.clamp(tensor * std + mean, 0, 1)

# Download a test image from the web
# Using a cat image from Wikimedia Commons (public domain)
TEST_IMAGE_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/4d/Cat_November_2010-1a.jpg/320px-Cat_November_2010-1a.jpg"

print("Downloading test image...")
try:
    with urllib.request.urlopen(TEST_IMAGE_URL) as response:
        test_image = Image.open(io.BytesIO(response.read())).convert('RGB')
    print(f"Downloaded image: {test_image.size} pixels")
except Exception:
    # Generate a synthetic test image if download fails
    print("Download failed. Generating synthetic test image.")
    img_array = np.random.randint(100, 200, (320, 320, 3), dtype=np.uint8)
    test_image = Image.fromarray(img_array)

# Apply preprocessing
input_tensor = preprocess(test_image).unsqueeze(0)  # Add batch dimension
print(f"\nPreprocessed tensor shape: {input_tensor.shape}")
print(f"  (batch=1, channels=3, height=224, width=224)")
print(f"  Value range: [{input_tensor.min():.3f}, {input_tensor.max():.3f}]")

# Visualize the preprocessing steps
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].imshow(test_image.resize((224, 224)))
axes[0].set_title("Original Image (resized for display)")
axes[0].axis('off')

# Show preprocessed image after denormalization
display_tensor = denormalize(input_tensor.squeeze(0))
axes[1].imshow(display_tensor.permute(1, 2, 0).numpy())
axes[1].set_title("After Preprocessing (denormalized for display)")
axes[1].axis('off')

plt.suptitle("Image Preprocessing Pipeline Verification", fontsize=14)
plt.tight_layout()
plt.show()

print("\nPreprocessing pipeline verified.")

## 6. Captum Smoke Test

The most important test: verify that Captum's core attribution pipeline runs end-to-end without errors. This tests import, model forward pass, gradient computation, and attribution output shape.

In [ ]:
# Full Captum smoke test
import torch
from captum.attr import IntegratedGradients, Saliency, LayerGradCam
from captum.attr import visualization as viz

print("Running Captum smoke tests...\n")

# Prepare input
# Enable gradients — required for all gradient-based Captum methods
input_for_attr = input_tensor.requires_grad_(True)

# Get the model's prediction first
resnet50.eval()
with torch.no_grad():
    logits = resnet50(input_tensor)
    probs = torch.softmax(logits, dim=1)
    top_class = probs.argmax().item()
    confidence = probs.max().item()

print(f"Model prediction: {imagenet_labels[top_class]} (class {top_class})")
print(f"Confidence: {confidence:.2%}")
print(f"\nRunning attributions for predicted class {top_class}...")

# Test 1: Saliency (fastest gradient method)
saliency = Saliency(resnet50)
sal_attr = saliency.attribute(input_for_attr, target=top_class)
print(f"\n[1] Saliency:")
print(f"    Output shape: {sal_attr.shape}")
print(f"    Value range: [{sal_attr.min():.4f}, {sal_attr.max():.4f}]")
assert sal_attr.shape == input_tensor.shape, "Shape mismatch!"
print("    PASSED")

# Test 2: Integrated Gradients (the axiomatic standard)
ig = IntegratedGradients(resnet50)
baseline = torch.zeros_like(input_tensor)  # Zero (black) baseline
ig_attr, delta = ig.attribute(
    input_for_attr,
    baselines=baseline,
    target=top_class,
    n_steps=50,
    return_convergence_delta=True
)
print(f"\n[2] Integrated Gradients:")
print(f"    Output shape: {ig_attr.shape}")
print(f"    Convergence delta: {delta.item():.5f} (should be near 0)")
assert ig_attr.shape == input_tensor.shape, "Shape mismatch!"
print("    PASSED")

# Test 3: LayerGradCam (layer attribution)
target_layer = resnet50.layer4[-1]  # Last residual block
lgc = LayerGradCam(resnet50, target_layer)
lgc_attr = lgc.attribute(input_for_attr, target=top_class)
print(f"\n[3] LayerGradCam:")
print(f"    Output shape: {lgc_attr.shape}")
print(f"    (Spatial map at layer4 resolution: 7×7 for 224×224 input)")
print("    PASSED")

print("\n" + "=" * 60)
print("All Captum smoke tests PASSED")
print("Your environment is ready for the course.")
print("=" * 60)

## 7. Quick Attribution Visualization

A preview of what the course produces: a side-by-side comparison of the input image and its Integrated Gradients attribution heatmap.

In [ ]:
# Quick visualization preview
import numpy as np
import matplotlib.pyplot as plt
from captum.attr import visualization as viz

# Prepare arrays for visualization
# Captum's viz module expects numpy arrays in (H, W, C) format
image_np = denormalize(input_tensor.squeeze(0)).permute(1, 2, 0).numpy()

# For multi-channel attribution (C, H, W), take the mean absolute value across channels
# This collapses RGB attribution to a single importance score per pixel
attr_np = ig_attr.squeeze(0).permute(1, 2, 0).detach().numpy()

fig, axes = viz.visualize_image_attr_multiple(
    attr_np,
    image_np,
    methods=["original_image", "heat_map", "blended_heat_map"],
    signs=["all", "absolute_value", "absolute_value"],
    titles=[
        f"Original Image",
        f"IG Attribution Heatmap",
        f"Attribution Overlay"
    ],
    show_colorbar=True,
    fig_size=(15, 4),
    use_pyplot=True
)

plt.suptitle(
    f"Integrated Gradients — Prediction: {imagenet_labels[top_class]} ({confidence:.1%})",
    fontsize=13,
    y=1.02
)
plt.tight_layout()
plt.show()

print(f"\nWarm colors = regions that increased the prediction of '{imagenet_labels[top_class]}'")
print("Cool colors = regions that decreased the prediction")
print("\nThis is Integrated Gradients. You will understand exactly how it works in Module 02.")

## Summary

### Environment Status

If all cells ran without errors, your environment is fully configured:

- **PyTorch**: Installed and hardware-aware
- **torchvision**: Pretrained CNN models available
- **Captum**: Attribution pipeline working end-to-end
- **transformers**: HuggingFace models accessible
- **Visualization**: matplotlib and Captum viz confirmed

### What's Next

**Notebook 02** produces a complete, annotated explanation of an ImageNet prediction using Integrated Gradients — the most principled gradient-based attribution method. You will understand every line by the end of Module 02.

### Troubleshooting

| Problem | Solution |
|---------|----------|
| `ImportError: captum` | `pip install captum` then restart kernel |
| CUDA out of memory | Reduce batch size or set `DEVICE = 'cpu'` |
| Model download fails | Check internet connection; models are ~100MB each |
| `requires_grad` error | Ensure input tensor has `requires_grad_(True)` |
| Image download fails | Replace with any local JPEG/PNG using `Image.open(path)` |